In [1]:
import torch
import gc

# Delete the model and processor variables if they exist
try:
    del model
    del processor
except NameError:
    pass

# Force Python's garbage collector to run
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()

In [1]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

Hugging Face cache set to: /workspace/huggingface_cache


In [2]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel
from qwen_vl_utils import process_vision_info

# 1. Setup paths
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
adapter_path = "/workspace/qwen_lora_full_run/checkpoint-100"
test_image_path = "/workspace/test_bar_2.png" # Pick one NOT in your 300 batch

# 2. Load Processor and Base Model
processor = AutoProcessor.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=["visual"] 
)

print("Loading base model...")
base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

# 3. Attach your trained LoRA adapter
print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval() # Crucial: set to evaluation mode

# 4. Prepare the test prompt
# 4. Prepare the test prompt
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image", 
                "image": test_image_path,
                "max_pixels": 768 * 768 # Force the same compression used during training
            },
            {"type": "text", "text": "Extract the Extended ChartSpec from this chart image. Return only the valid JSON."}
        ]
    }
]

# Apply chat template and process vision inputs
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info([messages])

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

# 5. Generate the JSON
print("Generating Extended ChartSpec...")
with torch.no_grad():
    generated_ids = model.generate(
        **inputs, 
        max_new_tokens=4096, # Give it enough room for the full JSON
        temperature=0.1,     # Keep it low for structured JSON tasks
        do_sample=False      # Greedy decoding is best for exact formatting
    )

# 6. Clean up and print the output
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_json = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print("\n=== MODEL OUTPUT ===")
print(output_json)

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Attaching LoRA adapter...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating Extended ChartSpec...

=== MODEL OUTPUT ===
{"title":"Vertical Grouped Bar Chart","topo":{"type":"bar","sub":"vertical"},"legend":{"on":1},"axes":[{"name":"x_axis","scl":"categorical","dom":["2018","2019","2020","2021"]},{"name":"y_axis","lbl":"$-","scl":"linear","dom":[0.0,270000]}],"ser":[{"name":"Canada","color":"#1e90ff","y_ref":"y_axis","data":[[2018,110000],[2019,135000],[2020,95000],[2021,140000]],"trend":{"global_trend":"fluctuating","rate_of_change":"rapid","intervals":[{"x_range":[2018,2020],"trend":"peak_then_drop"},{"x_range":[2020,2021],"trend":"increase"}]},"stats":{"max_value":140000,"max_at_the_ref_point":2021,"min_value":95000,"min_at_the_ref_point":2020,"threshold_coverage":{"above_mean":1,"all_positive":1,"hits_zero":0}}},{"name":"USA","color":"#ff7f50","y_ref":"y_axis","data":[[2018,230000],[2019,195000],[2020,165000],[2021,220000]],"trend":{"global_trend":"fluctuating","rate_of_change":"rapid","intervals":[{"x_range":[2018,2020],"trend":"peak_then_drop"}

In [ ]:
import json
import re

def validate_model_output(raw_output_text):
    print("--- Validation Check ---")
    
    # 1. Strip out any markdown formatting the model might hallucinate
    # (Sometimes models wrap outputs in ```json ... ```)
    clean_text = raw_output_text.strip()
    if clean_text.startswith("```json"):
        clean_text = clean_text[7:]
    if clean_text.startswith("```"):
        clean_text = clean_text[3:]
    if clean_text.endswith("```"):
        clean_text = clean_text[:-3]
        
    clean_text = clean_text.strip()
    
    # 2. Attempt strict JSON parsing
    try:
        parsed_json = json.loads(clean_text)
        print("✅ SUCCESS: Output is 100% syntactically valid JSON.")
        
        # 3. Check for your required top-level keys
        required_keys = ["title", "topo", "axes", "ser"]
        missing_keys = [key for key in required_keys if key not in parsed_json]
        
        if not missing_keys:
            print("✅ SUCCESS: All core Extended ChartSpec keys are present.")
        else:
            print(f"⚠️ WARNING: Valid JSON, but missing core keys: {missing_keys}")
            
        return parsed_json
            
    except json.JSONDecodeError as e:
        print("❌ FAILED: Output is NOT valid JSON.")
        print(f"Error Details: {e}")
        
        # Helper to show exactly where it broke
        error_line = clean_text.splitlines()[e.lineno - 1]
        print(f"Broken line ({e.lineno}): {error_line}")
        print(f"{' ' * (e.colno + 14)}^--- Error roughly here")
        return None

# Test the output from your inference script
validated_spec = validate_model_output(output_json)